# Devoir 1 - NLP pipeline for the Moroccan Code de la Route


In [22]:
from pathlib import Path
import re

import pandas as pd
import PyPDF2

try:
    from pyarabic.araby import strip_tashkeel
except ImportError:
    strip_tashkeel = None
    print("Install pyarabic if needed: pip install pyarabic")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

## Step 1 - Read the PDF


In [23]:
pdf_path = Path("week2/code de la route MA52_05.pdf")

reader = PyPDF2.PdfReader(str(pdf_path))
raw_pages = []

for page_number, page in enumerate(reader.pages[:3], start=1):
    text = page.extract_text() or ""
    raw_pages.append({"page": page_number, "text": text})

raw_text = "\n".join(page["text"] for page in raw_pages)

print("Number of pages used:", len(raw_pages))
print(raw_text[:1000])

Number of pages used: 3
تم إعداد هذه النسخة من أجل تسهيل  
مقروئية النص، وال يحتج إال بالنصوص  
في صيغتها املنشورة بالجريدة الرسمية المتعلق 52.05  القانون رقم 
بمدونة السير على الطرق، 
كما وقع تغييره وتتميمه 
صيغة موطدة بتاريخ 
2024  يوليو10
1
األمانة العامة للحكومة 52.05 القانون رقم 
 1.10.07 املتعلق بمدونة السير على الطرق الصادر بتنفيذه الظهير الشريف رقم 
)،2010  فبراير 11( 1431  من صفر 26 بتاريخ 
كما وقع تغييره وتتميمه
)2168 : )، ص 2010  مارس 25(1431  ربيع اآلخر 8  بتاريخ 5824 ج.ر عدد( 
لكتاب األولا
شروط السير على الطريق العمومية
القسم األول
رخصة السياقة
الباب األول
إلزامية رخصة السياقة
1 املادة
ال يجوز ألي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية 
ما لم يكن حاصال على رخصة للسياقة سارية الصالحية ومسلمة من قبل اإلدارة، تناسب صنف 
املركبة أو مجموعة املركبات التي يسوقها.
2 املادة
استثناء من أحكام املادة األولى أعاله :
 - يجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها 1
 سنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السياقة املسلم

## Step 2 - Arabic normalization

In [24]:
def normalize_arabic(text: str) -> str:
    """Normalize Arabic text to make matching more stable."""
    if not isinstance(text, str):
        return ""

    if strip_tashkeel is not None:
        text = strip_tashkeel(text)

    replacements = {
        "\u200f": "", "\u200e": "", "\u0640": "",
        "\u0623": "\u0627", "\u0625": "\u0627", "\u0622": "\u0627",
        "\u0649": "\u064a", "\u0624": "\u0648", "\u0626": "\u064a", "\u0629": "\u0647",
        "\u0671": "\u0627",
        "\u0627\u0645\u0644": "\u0627\u0644\u0645",
        "\u0628\u0627\u0645\u0644": "\u0628\u0627\u0644\u0645",
        "\u0648\u0627\u0645\u0644": "\u0648\u0627\u0644\u0645",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

clean_text = normalize_arabic(raw_text)

print(clean_text[:1000])

تم اعداد هذه النسخه من اجل تسهيل 
مقروييه النص، وال يحتج اال بالنصوص 
في صيغتها المنشوره بالجريده الرسميه المتعلق 52.05 القانون رقم 
بمدونه السير علي الطرق، 
كما وقع تغييره وتتميمه 
صيغه موطده بتاريخ 
2024 يوليو10
1
االمانه العامه للحكومه 52.05 القانون رقم 
 1.10.07 المتعلق بمدونه السير علي الطرق الصادر بتنفيذه الظهير الشريف رقم 
)،2010 فبراير 11( 1431 من صفر 26 بتاريخ 
كما وقع تغييره وتتميمه
)2168 : )، ص 2010 مارس 25(1431 ربيع االخر 8 بتاريخ 5824 ج.ر عدد( 
لكتاب االولا
شروط السير علي الطريق العموميه
القسم االول
رخصه السياقه
الباب االول
الزاميه رخصه السياقه
1 الماده
ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجموعه مركبات علي الطريق العموميه 
ما لم يكن حاصال علي رخصه للسياقه ساريه الصالحيه ومسلمه من قبل االداره، تناسب صنف 
المركبه او مجموعه المركبات التي يسوقها.
2 الماده
استثناء من احكام الماده االولي اعاله :
 - يجوز للمغاربه القاطنين بالخارج ان يسوقوا، داخل التراب الوطني، خالل مده اقصاها 1
 سنه واحده ابتداء من اقامتهم بالمغرب، بواسطه رخصه السياقه المسلمه لهم بالخارج ساريه
الصالحيه ؛ 
 

## Step 3 - Split into articles



In [25]:
ARTICLE_WORD = "(?:\u0627\u0644\u0645\u0627\u062f[\u0629\u0647]|\u0645\u0627\u062f[\u0629\u0647])"
ARTICLE_PATTERN = re.compile(
    rf"(?:^|\n)\s*(?:{ARTICLE_WORD}\s*(\d{{1,3}})|(\d{{1,3}})\s*{ARTICLE_WORD})",
    flags=re.MULTILINE,
)
PAGE_HEADER_PATTERN = re.compile(r"\n?\s*\d+\s*\n?\s*\u0627\u0644\u0627\u0645\u0627\u0646\u0647\s+\u0627\u0644\u0639\u0627\u0645\u0647\s+\u0644\u0644\u062d\u0643\u0648\u0645\u0647\s*")


def split_articles(text: str) -> list[dict]:
    matches = list(ARTICLE_PATTERN.finditer(text))
    articles = []

    for index, match in enumerate(matches):
        article_id = match.group(1) or match.group(2)
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        body = PAGE_HEADER_PATTERN.sub(" ", body)
        body = re.sub(r"\s+", " ", body).strip()

        if body:
            articles.append({"article_id": article_id, "article_text": body})

    return articles

articles = split_articles(clean_text)

print("Articles found:", len(articles))
print(articles[0])

Articles found: 4
{'article_id': '1', 'article_text': 'ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجموعه مركبات علي الطريق العموميه ما لم يكن حاصال علي رخصه للسياقه ساريه الصالحيه ومسلمه من قبل االداره، تناسب صنف المركبه او مجموعه المركبات التي يسوقها.'}


## Step 4 - Rules-based NLP with Regex


In [26]:
def parse_number(value: str):
    if value is None:
        return None
    value = re.sub(r"[^0-9]", "", value)
    return int(value) if value else None


def extract_amounts(text: str) -> list[int]:
    dirham = "(?:\u062f\u0631\u0647\u0645|\u062f\u0631\u0627\u0647\u0645)"
    pattern = re.compile(rf"\(?\s*(\d[\d\.\s]{{0,15}})\s*\)?\s*{dirham}")
    amounts = []
    for match in pattern.finditer(text):
        number = parse_number(match.group(1))
        if number is not None:
            amounts.append(number)
    return amounts


def extract_points(text: str) -> list[int]:
    point_word = "(?:\u0646\u0642\u0637|\u0646\u0642\u0637\u0647)"
    patterns = [
        rf"\u062e\u0635\u0645\s+\(?\s*(\d{{1,2}})\s*\)?\s*{point_word}",
        rf"\(?\s*(\d{{1,2}})\s*\)?\s*{point_word}\s+\u0645\u0646\s+\u0631\u0635\u064a\u062f",
        rf"\(?\s*(\d{{1,2}})\s*\)?\s*{point_word}",
        rf"{point_word}[^0-9]{{0,12}}(\d{{1,2}})",
    ]
    points = []
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            # Some patterns have the point word first, so the digit may be in the last group.
            value = parse_number(next(group for group in match.groups() if group))
            if value is not None and 0 < value <= 30:
                points.append(value)

    word_numbers = {
        "\u0648\u0627\u062d\u062f": 1,
        "\u0648\u0627\u062d\u062f\u0647": 1,
        "\u0627\u062b\u0646\u064a\u0646": 2,
        "\u0627\u062b\u0646\u062a\u064a\u0646": 2,
        "\u0627\u062b\u0646\u062a\u064a": 2,
        "\u062b\u0644\u0627\u062b": 3,
        "\u0627\u0631\u0628\u0639": 4,
        "\u062e\u0645\u0633": 5,
        "\u0633\u062a": 6,
        "\u0633\u0628\u0639": 7,
        "\u062b\u0645\u0627\u0646": 8,
        "\u062a\u0633\u0639": 9,
        "\u0639\u0634\u0631": 10,
        "\u0627\u062d\u062f\u064a \u0639\u0634\u0631\u0647": 11,
        "\u0627\u062b\u0646\u062a\u064a \u0639\u0634\u0631\u0647": 12,
        "\u062b\u0644\u0627\u062b\u064a\u0646": 30,
    }
    for word, value in word_numbers.items():
        if re.search(rf"{word}\s+{point_word}", text):
            points.append(value)

    return sorted(set(points))


def classify_vehicle(text: str) -> str:
    vehicle_patterns = {
        "moto / deux roues": ["\u062f\u0631\u0627\u062c\u0647", "\u0646\u0627\u0631\u064a\u0647", "\u062f\u0631\u0627\u062c\u0627\u062a"],
        "poids lourd / marchandises": ["\u0646\u0642\u0644 \u0627\u0644\u0628\u0636\u0627\u0626\u0639", "\u0648\u0632\u0646\u0647\u0627", "\u0643\u064a\u0644\u0648\u063a\u0631\u0627\u0645", "\u0634\u0627\u062d\u0646\u0647", "26.000", "3.500"],
        "transport collectif": ["\u0627\u0644\u0646\u0642\u0644 \u0627\u0644\u062c\u0645\u0627\u0639\u064a", "\u062d\u0627\u0641\u0644\u0647", "\u0627\u0644\u0627\u0634\u062e\u0627\u0635", "\u0645\u0642\u0627\u0639\u062f"],
        "vehicule leger / voiture": ["\u0633\u064a\u0627\u0631\u0647", "\u0627\u0644\u0633\u064a\u0627\u0631\u0647", "\u0645\u0631\u0643\u0628\u0647", "\u0627\u0644\u0645\u0631\u0643\u0628\u0647"],
    }
    found = []
    for label, keywords in vehicle_patterns.items():
        if any(keyword in text for keyword in keywords):
            found.append(label)
    return "; ".join(found) if found else "non specifie"


def extract_keywords(text: str) -> list[str]:
    keyword_patterns = {
        "vitesse": ["\u0633\u0631\u0639\u0647", "\u0627\u0644\u0633\u0631\u0639\u0647"],
        "stationnement": ["\u0648\u0642\u0648\u0641", "\u062a\u0648\u0642\u0641", "\u0627\u0644\u062a\u0648\u0642\u0641", "\u0627\u0644\u0648\u0642\u0648\u0641"],
        "permis": ["\u0631\u062e\u0635\u0647 \u0627\u0644\u0633\u064a\u0627\u0642\u0647", "\u0631\u062e\u0635\u0647"],
        "ceinture": ["\u062d\u0632\u0627\u0645 \u0627\u0644\u0633\u0644\u0627\u0645\u0647"],
        "alcool": ["\u0643\u062d\u0648\u0644", "\u0633\u0643\u0631"],
        "accident": ["\u062d\u0627\u062f\u062b\u0647", "\u062d\u0627\u062f\u062b"],
        "telephone": ["\u0647\u0627\u062a\u0641", "\u0627\u0644\u0647\u0627\u062a\u0641"],
        "autoroute": ["\u0627\u0644\u0637\u0631\u064a\u0642 \u0627\u0644\u0633\u064a\u0627\u0631"],
        "priorite": ["\u0627\u0633\u0628\u0642\u064a\u0647", "\u0627\u0644\u0627\u0633\u0628\u0642\u064a\u0647"],
        "depassement": ["\u062a\u062c\u0627\u0648\u0632"],
        "chargement": ["\u062d\u0645\u0648\u0644\u0647", "\u0627\u0644\u0628\u0636\u0627\u0626\u0639", "\u0648\u0632\u0646"],
        "documents": ["\u0634\u0647\u0627\u062f\u0647 \u0627\u0644\u062a\u0633\u062c\u064a\u0644", "\u0648\u062b\u064a\u0642\u0647", "\u0627\u0644\u0648\u062b\u0627\u0626\u0642"],
    }

    tags = []
    for tag, patterns in keyword_patterns.items():
        if any(pattern in text for pattern in patterns):
            tags.append(tag)
    return tags


def classify_role(text: str) -> str:
    sanction_pattern = "\u064a\u0639\u0627\u0642\u0628|\u063a\u0631\u0627\u0645\u0647|\u062f\u0631\u0647\u0645|\u062e\u0635\u0645|\u062d\u0628\u0633|\u062a\u0648\u0642\u064a\u0641|\u0633\u062d\u0628|\u0627\u0644\u063a\u0627\u0621"
    obligation_pattern = "\u064a\u062c\u0628|\u0644\u0627 \u064a\u062c\u0648\u0632|\u064a\u0644\u0632\u0645|\u064a\u062a\u0639\u064a\u0646|\u0639\u0644\u0649 \u0643\u0644"
    definition_pattern = "\u064a\u0642\u0635\u062f|\u062a\u0639\u0646\u064a|\u064a\u0631\u0627\u062f|\u064a\u0639\u062a\u0628\u0631"

    if re.search(sanction_pattern, text):
        return "sanction"
    if re.search(obligation_pattern, text):
        return "obligation"
    if re.search(definition_pattern, text):
        return "definition"
    return "autre"


def build_rule_row(article: dict) -> dict:
    text = article["article_text"]
    amounts = extract_amounts(text)
    points = extract_points(text)
    keywords = extract_keywords(text)

    return {
        "article_id": article["article_id"],
        "infraction_desc": text,
        "categorie_vehicule": classify_vehicle(text),
        "amende_fixe": max(amounts) if amounts else None,
        "amende_min": min(amounts) if amounts else None,
        "amende_max": max(amounts) if amounts else None,
        "points_retrait": max(points) if points else None,
        "mots_cles": ", ".join(keywords),
        "role_paragraphe": classify_role(text),
        "has_prison": bool(re.search("\u062d\u0628\u0633|\u0627\u0644\u0633\u062c\u0646", text)),
        "has_license_penalty": bool(re.search("\u062a\u0648\u0642\u064a\u0641|\u0633\u062d\u0628|\u0627\u0644\u063a\u0627\u0621", text)),
        "source": pdf_path.name,
    }

rows = [build_rule_row(article) for article in articles]
df_rules = pd.DataFrame(rows)

print(df_rules.shape)
df_rules.head()

(4, 12)


,article_id,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,points_retrait,mots_cles,role_paragraphe,has_prison,has_license_penalty,source
0,1,ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجمو...,vehicule leger / voiture,None,None,None,None,permis,autre,False,False,code de la route MA52_05.pdf
1,2,استثناء من احكام الماده االولي اعاله : - يجوز ...,non specifie,None,None,None,None,permis,obligation,False,False,code de la route MA52_05.pdf
2,4,في حاله السير الدولي ووفقا لالتفاقيه الدوليه ل...,non specifie,None,None,None,None,"permis, depassement",autre,False,False,code de la route MA52_05.pdf
3,5,من 13 بتاريخ 1.16.106 الصادر بتنفيذه الظهري ال...,non specifie,None,None,None,None,"permis, alcool",autre,False,False,code de la route MA52_05.pdf


## Step 5 - NLP model / clustering


In [27]:
def add_nlp_clusters(df: pd.DataFrame, n_clusters: int = 6) -> pd.DataFrame:
    result = df.copy()
    texts = result["infraction_desc"].fillna("").tolist()
    n_clusters = min(n_clusters, max(1, len(texts)))

    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        vectors = model.encode(texts, show_progress_bar=True)
        result["model_approach"] = "pretrained_sentence_transformer"
    except Exception as exc:
        print("Pretrained model unavailable, using TF-IDF fallback:", exc)
        vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
        vectors = vectorizer.fit_transform(texts)
        result["model_approach"] = "tfidf_fallback"

    if len(texts) > 1:
        model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        result["cluster_theme"] = model.fit_predict(vectors)
    else:
        result["cluster_theme"] = 0

    return result

final_df = add_nlp_clusters(df_rules, n_clusters=6)

final_df[["article_id", "role_paragraphe", "amende_fixe", "points_retrait", "mots_cles", "cluster_theme", "model_approach"]].head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,article_id,role_paragraphe,amende_fixe,points_retrait,mots_cles,cluster_theme,model_approach
0,1,autre,None,None,permis,2,pretrained_sentence_transformer
1,2,obligation,None,None,permis,0,pretrained_sentence_transformer
2,4,autre,None,None,"permis, depassement",1,pretrained_sentence_transformer
3,5,autre,None,None,"permis, alcool",3,pretrained_sentence_transformer


## Step 6 - Export final CSV

In [28]:
output_path = Path("export_final.csv")
final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Generated file:", output_path.resolve())
print("Rows:", len(final_df))
print("Columns:", list(final_df.columns))

Generated file: C:\Users\amami\OneDrive\Documents\GitHub\nlp week 2\export_final.csv
Rows: 4
Columns: ['article_id', 'infraction_desc', 'categorie_vehicule', 'amende_fixe', 'amende_min', 'amende_max', 'points_retrait', 'mots_cles', 'role_paragraphe', 'has_prison', 'has_license_penalty', 'source', 'model_approach', 'cluster_theme']
